# Cross-Modality Unsupervised Domain Adaptation: NIH ChestX-ray14 (CXR, source) -> MosMedData Chest CT (target)

**Görev:** NIH ChestX-ray14 üzerinde egitilen binary hastalik siniflandiricisini
(Pneumonia/Infiltration vs No Finding), etiketleri egitim sirasinda kullanilmayan
MosMedData Chest CT (CT-0 vs CT-1..4) domainine DANN (Domain-Adversarial Neural
Network) ile Unsupervised Domain Adaptation (UDA) yontemiyle transfer eder ve
Source-Only baseline ile karsilastirir.

## Protokol notlari (ozet - tam Methodology Review sohbette verildi)

1. **Label semantiği uyumsuzlugu:** NIH `Pneumonia`/`Infiltration` genel/heterojen
   pnomoni bulgularidir; MosMed `CT-1..4` ozellikle **COVID-19** siddet
   derecelendirmesidir. Bu nedenle bu deney, salt bir "modality gap" degil, ayni
   zamanda kismi bir **task/populasyon kaymasi** icerir. Sonuclar "COVID-19 tanisinin
   CXR'dan CT'ye transferi" olarak degil, "akciger anormalligi / opasite tespiti
   yeteneginin CXR->CT modalite ve populasyon kaymasi altinda transferi" olarak
   yorumlanmalidir.
2. **Negatif sinif esdegerligi kismi:** `No Finding` (genel hastane CXR populasyonu)
   ile `CT-0` (COVID supheli tarama populasyonunda normal CT) ayni "patoloji yok"
   kavramini paylasir, ancak alttaki hasta populasyonlari farklidir (selection bias
   riski).
3. **2D CXR vs 3D CT temsili:** Tek bir CXR projeksiyonu tum toraksi ozetler; her CT
   slice ise ince bir aksiyal kesittir. Bu yapisal fark, volume-level cok-slice
   agregasyonuyla (bkz. Section 14) hafifletilir ama tamamen ortadan kalkmaz.
4. **Model selection icin target validation etiketi kullanimi**
   (`USE_TARGET_LABELS_FOR_MODEL_SELECTION=True`), saf UDA protokolunden bir sapmadir
   (yalniz checkpoint secimi icin, egitim gradyanlarina hicbir target etiketi
   girmez). Bu acikca raporlanir; tamamen unsupervised alternatif de
   (`USE_TARGET_LABELS_FOR_MODEL_SELECTION=False`, source validation loss'a dayali)
   koddadir.
5. **Negative transfer riski:** DANN, buyuk modalite/task farki nedeniyle,
   class-discriminative bilgiyi feda ederek domain-invariant (ama ayirt edici olmayan)
   feature'lar ogrenebilir. Bu yuzden `domain accuracy ~0.50` ve dusen MMD/CORAL TEK
   BASINA basari kaniti degildir; Section 13, 17 ve 19'daki yorumlara bakiniz.
6. NIH'de `No Finding`/`Pneumonia`/`Infiltration` disindaki diger bulgu etiketleri
   (orn. sadece `Cardiomegaly`) bu binary görevde **belirsiz** kabul edilip
   veri setinden cikarilir (Section 4, `build_nih_labels`).

Bu notebook Kaggle GPU ortaminda calisacak sekilde tasarlanmistir; tum fonksiyonlar
once TANIMLANIR (Section 1-18), gercek veri kesfi/egitim/degerlendirme ise
Section 19 (Experiment runner) icinde CAGRILIR.


In [ ]:
# Kaggle varsayilan imajinda nibabel bulunmayabilir; scikit-learn/torch/torchvision genelde hazirdir.
import sys
import importlib
if importlib.util.find_spec("nibabel") is None:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)


## 1. Imports


In [ ]:
import os
import sys
import glob
import math
import random
import warnings
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.autograd import Function
from torchvision import transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    f1_score, balanced_accuracy_score, confusion_matrix, roc_curve,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Configuration


In [ ]:
@dataclass
class Config:
    # ---------------- Paths (Kaggle input dataset yollariniza gore duzenleyin) ----------------
    NIH_ROOT: str = "BURAYA_NIH_DATASET_PATH"
    NIH_CSV: str = "BURAYA_DATA_ENTRY_2017_CSV_PATH"
    MOSMED_ROOT: str = "BURAYA_MOSMED_DATASET_PATH"
    OUTPUT_DIR: str = "/kaggle/working"

    # ---------------- Reproducibility ----------------
    SEED: int = 42

    # ---------------- Source (NIH) label tanimi ----------------
    # True  -> Positive = Pneumonia OR Infiltration, Negative = No Finding
    # False -> Positive = Pneumonia,                 Negative = No Finding
    INCLUDE_INFILTRATION: bool = True

    # ---------------- Source class imbalance ----------------
    USE_WEIGHTED_SAMPLER: bool = True   # True: WeightedRandomSampler; False: weighted CrossEntropyLoss

    # ---------------- NIH train/val split (source-domain model selection icin) ----------------
    NIH_VAL_RATIO: float = 0.10

    # ---------------- MosMed (target) split oranlari ----------------
    TARGET_TRAIN_RATIO: float = 0.70
    TARGET_VAL_RATIO: float = 0.10
    TARGET_TEST_RATIO: float = 0.20

    # ---------------- CT preprocessing ----------------
    CT_HU_MIN: float = -1000.0
    CT_HU_MAX: float = 400.0
    CT_CENTRAL_SLICE_MIN_FRAC: float = 0.30   # random/eval slice secimi icin merkezi bolge alt sinir
    CT_CENTRAL_SLICE_MAX_FRAC: float = 0.70   # merkezi bolge ust sinir
    NUM_EVAL_SLICES: int = 7                  # validation/test icin volume basina slice sayisi

    # ---------------- Goruntu ----------------
    IMG_SIZE: int = 224

    # ---------------- Model ----------------
    BACKBONE_NAME: str = "resnet18"   # get_backbone() ile kolayca degistirilebilir (resnet18/34/50)
    NUM_CLASSES: int = 2

    # ---------------- Egitim ----------------
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2
    NUM_EPOCHS_SOURCE_ONLY: int = 15
    NUM_EPOCHS_DANN: int = 15
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    DOMAIN_LOSS_WEIGHT: float = 1.0
    USE_AMP: bool = True

    # ---------------- Model selection ----------------
    # True  -> MosMed VALIDATION AUC (target etiketleri YALNIZCA checkpoint secimi icin kullanilir;
    #          egitim gradyanlarina target etiketi hicbir sekilde girmez).
    # False -> Tamamen unsupervised: NIH source validation loss.
    USE_TARGET_LABELS_FOR_MODEL_SELECTION: bool = True

    # ---------------- Domain alignment analizi ----------------
    FEATURE_EXTRACTION_N_SAMPLES: int = 200

    # ---------------- Checkpoint / sonuc yollari ----------------
    SOURCE_ONLY_CKPT_PATH: str = "/kaggle/working/source_only_best.pth"
    DANN_CKPT_PATH: str = "/kaggle/working/dann_best.pth"
    RESULTS_CSV_PATH: str = "/kaggle/working/experiment_results.csv"

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    def validate(self):
        ratio_sum = self.TARGET_TRAIN_RATIO + self.TARGET_VAL_RATIO + self.TARGET_TEST_RATIO
        if not math.isclose(ratio_sum, 1.0, abs_tol=1e-6):
            raise ValueError(f"TARGET split oranlari toplami 1.0 olmali, mevcut: {ratio_sum}")
        if not (0.0 < self.NIH_VAL_RATIO < 1.0):
            raise ValueError(f"NIH_VAL_RATIO (0,1) araliginda olmali: {self.NIH_VAL_RATIO}")
        if self.CT_HU_MIN >= self.CT_HU_MAX:
            raise ValueError("CT_HU_MIN, CT_HU_MAX'tan kucuk olmalidir")


cfg = Config()
cfg.validate()
print(cfg)


## 3. Reproducibility


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(cfg.SEED)
print(f"Seed set: {cfg.SEED}")


## 4. Dataset discovery


In [ ]:
def discover_nih_images(nih_root: str) -> Dict[str, str]:
    """NIH_ROOT altindaki tum PNG/JPG dosyalarini dosya adina gore indeksler."""
    if not os.path.isdir(nih_root):
        raise FileNotFoundError(f"NIH_ROOT bulunamadi: {nih_root}")
    index = {}
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG"):
        for path in glob.glob(os.path.join(nih_root, "**", ext), recursive=True):
            index[os.path.basename(path)] = path
    if len(index) == 0:
        raise FileNotFoundError(f"NIH_ROOT altinda hic goruntu bulunamadi: {nih_root}")
    print(f"[NIH] Diskte indekslenen goruntu sayisi: {len(index)}")
    return index


def build_nih_labels(nih_csv: str, image_index: Dict[str, str], include_infiltration: bool) -> pd.DataFrame:
    """
    Binary label tanimi:
      Negative = 'No Finding'
      Positive = 'Pneumonia' (ve include_infiltration=True ise 'Infiltration')
    Bu iki grubun disinda kalan (sadece baska bulgu iceren) goruntuler bu binary
    gorev icin ANLAMSIZ/BELIRSIZ oldugundan veri setinden cikarilir.
    """
    if not os.path.isfile(nih_csv):
        raise FileNotFoundError(f"NIH_CSV bulunamadi: {nih_csv}")
    df = pd.read_csv(nih_csv)
    required_cols = {"Image Index", "Finding Labels"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"NIH_CSV beklenen kolonlari icermiyor: {missing}")

    positive_labels = {"Pneumonia"}
    if include_infiltration:
        positive_labels.add("Infiltration")

    def to_binary_label(finding_labels) -> Optional[int]:
        labels = set(str(finding_labels).split("|"))
        labels = {l.strip() for l in labels}
        if "No Finding" in labels:
            return 0
        if labels & positive_labels:
            return 1
        return None

    df["label"] = df["Finding Labels"].apply(to_binary_label)
    n_excluded = df["label"].isna().sum()
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    df["filepath"] = df["Image Index"].map(image_index)
    n_missing_files = df["filepath"].isna().sum()
    if n_missing_files > 0:
        print(f"[UYARI][NIH] {n_missing_files} goruntu CSV'de var ama diskte bulunamadi, atlaniyor.")
    df = df.dropna(subset=["filepath"]).reset_index(drop=True)

    print(f"[NIH] include_infiltration={include_infiltration} -> Positive tanimi: {sorted(positive_labels)} vs 'No Finding'")
    print(f"[NIH] Binary-gorev disi (belirsiz) oldugu icin cikarilan ornek sayisi: {n_excluded}")
    print(f"[NIH] Toplam kullanilabilir ornek: {len(df)}")
    dist = df["label"].value_counts().rename({0: "Negative (No Finding)", 1: "Positive"})
    print(f"[NIH] Sinif dagilimi:\n{dist}")
    return df[["filepath", "label"]].reset_index(drop=True)


def discover_mosmed_volumes(mosmed_root: str) -> pd.DataFrame:
    """
    MosMed 'studies/CT-0..CT-4' klasor yapisini tarar.
    Binary target label: CT-0 -> Negative(0), CT-1..CT-4 -> Positive(1).
    """
    studies_dir = os.path.join(mosmed_root, "studies")
    if not os.path.isdir(studies_dir):
        raise FileNotFoundError(f"MosMed 'studies' klasoru bulunamadi: {studies_dir}")

    class_to_label = {"CT-0": 0, "CT-1": 1, "CT-2": 1, "CT-3": 1, "CT-4": 1}
    records = []
    for class_name, label in class_to_label.items():
        class_path = os.path.join(studies_dir, class_name)
        if not os.path.isdir(class_path):
            print(f"[UYARI][MosMed] {class_path} bulunamadi, atlaniyor.")
            continue
        volume_files = sorted(
            glob.glob(os.path.join(class_path, "**", "*.nii"), recursive=True)
            + glob.glob(os.path.join(class_path, "**", "*.nii.gz"), recursive=True)
        )
        for vp in volume_files:
            records.append({"filepath": vp, "label": label, "source_class": class_name})

    if len(records) == 0:
        raise FileNotFoundError(f"MosMed altinda hic .nii/.nii.gz bulunamadi: {studies_dir}")

    df = pd.DataFrame(records).reset_index(drop=True)
    print(f"[MosMed] Toplam volume: {len(df)}")
    print(f"[MosMed] Klasor bazinda sayim:\n{df.groupby('source_class')['label'].count()}")
    dist = df["label"].value_counts().rename({0: "Negative (CT-0)", 1: "Positive (CT-1..4)"})
    print(f"[MosMed] Binary sinif dagilimi:\n{dist}")
    return df


## 5. Dataset split


In [ ]:
def split_nih(df: pd.DataFrame, val_ratio: float, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """NIH source domain icin stratified train/validation split (source model selection icin)."""
    train_df, val_df = train_test_split(df, test_size=val_ratio, stratify=df["label"], random_state=seed)
    for name, split_df in [("TRAIN", train_df), ("VAL", val_df)]:
        dist = split_df["label"].value_counts(normalize=True).rename({0: "Negative", 1: "Positive"})
        print(f"[NIH-{name}] n={len(split_df)} | sinif orani:\n{dist}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def split_mosmed(df: pd.DataFrame, train_ratio: float, val_ratio: float, test_ratio: float,
                  seed: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    MosMed icin STRATIFIED 3-parcali split (CT-0 vs CT-1..4 orani her split'te korunur).
    Domain adaptation sirasinda SADECE mosmed_train_df kullanilir; val/test volume'lari
    egitim sirasinda backbone veya domain classifier tarafindan hicbir sekilde gorulmez.
    """
    ratio_sum = train_ratio + val_ratio + test_ratio
    if not math.isclose(ratio_sum, 1.0, abs_tol=1e-6):
        raise ValueError(f"Split oranlari toplami 1.0 olmali, mevcut: {ratio_sum}")

    train_df, temp_df = train_test_split(df, train_size=train_ratio, stratify=df["label"], random_state=seed)
    relative_val_ratio = val_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(temp_df, train_size=relative_val_ratio,
                                        stratify=temp_df["label"], random_state=seed)

    for name, split_df in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
        dist = split_df["label"].value_counts(normalize=True).rename({0: "Negative", 1: "Positive"})
        print(f"[MosMed-{name}] n={len(split_df)} | sinif orani:\n{dist}")

    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True))


## 6. Dataset classes


In [ ]:
# ---------------- NIH (2D CXR) ----------------
class NIHChestXrayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        self.filepaths = df["filepath"].tolist()
        self.labels = df["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        try:
            image = Image.open(path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"NIH goruntusu okunamadi: {path} ({e})")
        image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


# ---------------- MosMed (3D CT) yardimci fonksiyonlar ----------------
def get_volume_shape(path: str) -> Tuple[int, ...]:
    """NIfTI header'indan (veriyi tam yuklemeden) volume shape'ini okur."""
    try:
        img = nib.load(path)
    except Exception as e:
        raise RuntimeError(f"CT volume acilamadi: {path} ({e})")
    return img.shape


def get_axial_axis(shape: Tuple[int, ...]) -> int:
    """
    Aksiyal (slice) eksen, MosMed'in tipik ~512x512xN NIfTI duzeninde en kucuk
    boyuta sahip eksendir; bu sayede farkli orientasyonlara karsi otomatik/saglam
    bir tahmin yapilir.
    """
    return int(np.argmin(shape))


def load_ct_slice(path: str, axis: int, slice_idx: int) -> np.ndarray:
    """
    Tum volume'u belleğe yuklemeden (nibabel ArrayProxy uzerinden), SADECE istenen
    axial slice'i okur. NaN/Inf degerler HU sinirlarina clip edilerek temizlenir.
    """
    try:
        img = nib.load(path)
        data_proxy = img.dataobj
        slicer = [slice(None)] * data_proxy.ndim
        slicer[axis] = slice_idx
        slice_2d = np.asarray(data_proxy[tuple(slicer)], dtype=np.float32)
    except Exception as e:
        raise RuntimeError(f"CT slice okunamadi: {path} (axis={axis}, idx={slice_idx}) ({e})")
    slice_2d = np.nan_to_num(slice_2d, nan=0.0, posinf=cfg.CT_HU_MAX, neginf=cfg.CT_HU_MIN)
    return slice_2d


def apply_lung_window(slice_2d: np.ndarray, hu_min: float, hu_max: float) -> np.ndarray:
    """HU degerlerini akciger penceresine (orn. -1000..400) clip edip [0,1]'e normalize eder."""
    slice_2d = np.clip(slice_2d, hu_min, hu_max)
    slice_2d = (slice_2d - hu_min) / (hu_max - hu_min + 1e-8)
    return slice_2d.astype(np.float32)


def slice_to_rgb_pil(slice_2d: np.ndarray) -> Image.Image:
    """[0,1] grayscale slice'i ImageNet-pretrained backbone icin 3 kanalli PIL goruntusune cevirir."""
    slice_uint8 = (np.clip(slice_2d, 0.0, 1.0) * 255).astype(np.uint8)
    rgb = np.stack([slice_uint8] * 3, axis=-1)
    return Image.fromarray(rgb)


def sample_random_central_slice_idx(num_slices: int, min_frac: float, max_frac: float) -> int:
    """Volume'un merkezi (orn. %30-%70) bolgesinden RASTGELE bir slice indeksi secer."""
    lo = max(0, int(round(num_slices * min_frac)))
    hi = min(num_slices - 1, int(round(num_slices * max_frac)))
    if hi <= lo:
        lo, hi = 0, max(0, num_slices - 1)
    return random.randint(lo, hi)


def get_eval_slice_indices(num_slices: int, min_frac: float, max_frac: float, num_eval_slices: int) -> List[int]:
    """Merkezi akciger bolgesine ESIT ARALIKLARLA dagitilmis deterministik slice indeksleri uretir."""
    lo = max(0, int(round(num_slices * min_frac)))
    hi = min(num_slices - 1, int(round(num_slices * max_frac)))
    if hi <= lo:
        mid = num_slices // 2
        return [mid] * num_eval_slices
    raw_indices = np.linspace(lo, hi, num=num_eval_slices)
    indices = [int(round(i)) for i in raw_indices]
    return indices


# ---------------- MosMed (3D CT) Dataset siniflari ----------------
class MosMedCTTrainDataset(Dataset):
    """
    Egitim (DANN target-domain) icin: her __getitem__ cagrisinda volume'un merkezi
    bolgesinden RASTGELE bir axial slice secilir.

    ONEMLI (slice-caching hatasindan kacinma): secilen slice indeksi HICBIR YERDE
    (volume path -> slice) seklinde cache'lenmez; load_ct_slice() her cagrida ilgili
    slice'i nibabel'in lazy ArrayProxy'si uzerinden dogrudan diskten okur. Boylece
    (a) her epoch'ta ayni volume icin farkli bir slice gorulebilir (augmentation
    etkisi saglar) ve (b) tum volume RAM'e onbelleklenmedigi icin bellek kullanimi
    kontrol altinda kalir.
    """
    def __init__(self, df: pd.DataFrame, transform: Callable, config: "Config"):
        self.filepaths = df["filepath"].tolist()
        self.labels = df["label"].tolist()  # DANN target-domain training'de KULLANILMAZ (yalniz domain loss hesaplanir)
        self.transform = transform
        self.cfg = config

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        shape = get_volume_shape(path)
        axis = get_axial_axis(shape)
        num_slices = shape[axis]
        slice_idx = sample_random_central_slice_idx(
            num_slices, self.cfg.CT_CENTRAL_SLICE_MIN_FRAC, self.cfg.CT_CENTRAL_SLICE_MAX_FRAC
        )
        slice_2d = load_ct_slice(path, axis, slice_idx)
        slice_2d = apply_lung_window(slice_2d, self.cfg.CT_HU_MIN, self.cfg.CT_HU_MAX)
        image = slice_to_rgb_pil(slice_2d)
        image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


class MosMedCTEvalDataset(Dataset):
    """
    Validation/Test icin: her volume'dan NUM_EVAL_SLICES adet, merkezi akciger
    bolgesine esit araliklarla dagitilmis DETERMINISTIK slice cikarilir. Bu slice
    secimi rastgele olmadigi icin epoch'lar arasi tutarlilik bir sorun teskil etmez.
    Volume-level prediction = mean(slice_probabilities) Section 14'te hesaplanir.
    """
    def __init__(self, df: pd.DataFrame, transform: Callable, config: "Config"):
        self.transform = transform
        self.cfg = config
        self.filepaths = df["filepath"].tolist()
        self.labels = df["label"].tolist()
        self.items: List[Tuple[int, str, int, int]] = []  # (volume_idx, path, axis, slice_idx)
        for v_idx, path in enumerate(self.filepaths):
            shape = get_volume_shape(path)
            axis = get_axial_axis(shape)
            num_slices = shape[axis]
            slice_indices = get_eval_slice_indices(
                num_slices, config.CT_CENTRAL_SLICE_MIN_FRAC, config.CT_CENTRAL_SLICE_MAX_FRAC,
                config.NUM_EVAL_SLICES,
            )
            for s_idx in slice_indices:
                self.items.append((v_idx, path, axis, s_idx))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        v_idx, path, axis, s_idx = self.items[idx]
        slice_2d = load_ct_slice(path, axis, s_idx)
        slice_2d = apply_lung_window(slice_2d, self.cfg.CT_HU_MIN, self.cfg.CT_HU_MAX)
        image = slice_to_rgb_pil(slice_2d)
        image = self.transform(image)
        label = torch.tensor(self.labels[v_idx], dtype=torch.long)
        return image, label, v_idx

    def num_volumes(self) -> int:
        return len(self.filepaths)


## 7. Transforms


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_source_train_transform(config: "Config") -> transforms.Compose:
    """Source (CXR) egitim augmentation: makul geometrik bozulmalar."""
    return transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=7),
        transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_target_train_transform(config: "Config") -> transforms.Compose:
    """
    Target (CT) egitim augmentation: CXR pipeline'i KORU KORUNE kullanilmaz.
    Buyuk rotasyon/affine/shear gibi agresif geometrik bozulmalar CT kesitlerinde
    anatomik olarak gercekci olmayan goruntuler yaratabilir; bu yuzden yalnizca
    hafif bir yatay flip uygulanir.
    """
    return transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_eval_transform(config: "Config") -> transforms.Compose:
    """Validation/Test icin DETERMINISTIK transform (rastgelelik yok)."""
    return transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


## 8. DataLoaders


In [ ]:
def compute_class_sample_weights(labels: List[int]) -> np.ndarray:
    labels_arr = np.array(labels)
    class_counts = np.bincount(labels_arr, minlength=2).astype(np.float64)
    class_weights = 1.0 / np.clip(class_counts, a_min=1, a_max=None)
    return class_weights[labels_arr]


def compute_class_weights_tensor(labels: List[int]) -> torch.Tensor:
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    weights = counts.sum() / (2.0 * np.clip(counts, 1, None))
    return torch.tensor(weights, dtype=torch.float32)


def build_nih_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame,
                           config: "Config") -> Tuple[DataLoader, DataLoader]:
    train_transform = build_source_train_transform(config)
    eval_transform = build_eval_transform(config)

    train_ds = NIHChestXrayDataset(train_df, train_transform)
    val_ds = NIHChestXrayDataset(val_df, eval_transform)

    if config.USE_WEIGHTED_SAMPLER:
        sample_weights = compute_class_sample_weights(train_df["label"].tolist())
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=sampler,
                                   num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                                   num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)

    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=config.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader


def build_mosmed_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame,
                              config: "Config") -> Tuple[DataLoader, DataLoader, DataLoader]:
    target_train_transform = build_target_train_transform(config)
    eval_transform = build_eval_transform(config)

    train_ds = MosMedCTTrainDataset(train_df, target_train_transform, config)
    val_ds = MosMedCTEvalDataset(val_df, eval_transform, config)
    test_ds = MosMedCTEvalDataset(test_df, eval_transform, config)

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                               num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=config.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                              num_workers=config.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


## 9. Models


In [ ]:
def get_backbone(name: str, pretrained: bool = True) -> Tuple[nn.Module, int]:
    """
    Backbone'u kolayca degistirebilmek icin ayri bir factory fonksiyonu.
    (backbone_without_fc, feature_dim) dondurur. Yeni bir torchvision backbone
    eklemek icin sadece bu fonksiyona bir 'elif' dali eklemek yeterlidir.
    """
    name = name.lower()
    if name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        feature_dim = model.fc.in_features
    elif name == "resnet34":
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        model = resnet34(weights=weights)
        feature_dim = model.fc.in_features
    elif name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        feature_dim = model.fc.in_features
    else:
        raise ValueError(f"Desteklenmeyen backbone: {name}")

    model.fc = nn.Identity()
    return model, feature_dim


class ClassifierHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class DomainClassifierHead(nn.Module):
    """2 domain: source=0, target=1."""
    def __init__(self, in_features: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


class SourceOnlyModel(nn.Module):
    """Experiment 1: yalnizca backbone + classifier (domain adaptation bileseni yok)."""
    def __init__(self, config: "Config"):
        super().__init__()
        self.backbone, feat_dim = get_backbone(config.BACKBONE_NAME, pretrained=True)
        self.classifier = ClassifierHead(feat_dim, config.NUM_CLASSES)

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits, features


class DANNModel(nn.Module):
    """
    Experiment 2: Image -> Backbone -> {Classifier -> disease prediction,
                                          GRL -> Domain Classifier}
    """
    def __init__(self, config: "Config"):
        super().__init__()
        self.backbone, feat_dim = get_backbone(config.BACKBONE_NAME, pretrained=True)
        self.classifier = ClassifierHead(feat_dim, config.NUM_CLASSES)
        self.domain_classifier = DomainClassifierHead(feat_dim)

    def forward(self, x, grl_lambda: float = 0.0):
        features = self.backbone(x)
        class_logits = self.classifier(features)
        reversed_features = GradientReversalFunction.apply(features, grl_lambda)
        domain_logits = self.domain_classifier(reversed_features)
        return class_logits, domain_logits, features


## 10. Gradient Reversal Layer


In [ ]:
class GradientReversalFunction(Function):
    """Forward: identity. Backward: gradyani ters cevirip lambda ile olcekler."""
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None


def compute_grl_lambda(current_step: int, total_steps: int) -> float:
    """
    Klasik DANN lambda schedule'i (Ganin & Lempitsky, 2016):
        lambda_p = 2 / (1 + exp(-10 * p)) - 1
    p, yalnizca epoch seviyesinde degil, BATCH bazinda toplam training progress
    (current_step / total_steps) uzerinden hesaplanir.
    """
    p = current_step / max(total_steps, 1)
    p = min(max(p, 0.0), 1.0)
    return 2.0 / (1.0 + math.exp(-10 * p)) - 1.0


## 11. Training utilities


In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0

    def update(self, value: float, n: int = 1):
        self.sum += value * n
        self.count += n

    @property
    def avg(self) -> float:
        return self.sum / max(self.count, 1)


def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer,
                     epoch: int, config: "Config", validation_metrics: Dict):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "config": asdict(config),
        "validation_metrics": validation_metrics,
    }
    torch.save(checkpoint, path)
    print(f"[Checkpoint] Kaydedildi: {path} (epoch={epoch})")


def load_checkpoint(path: str, model: nn.Module, optimizer: Optional[torch.optim.Optimizer] = None,
                     device: str = "cpu") -> Dict:
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint


def evaluate_classifier_on_loader(forward_fn: Callable[[torch.Tensor], torch.Tensor], loader: DataLoader,
                                   device: str, use_amp: bool) -> Dict:
    """Generic 2D (image, label) loader uzerinde loss/AUC hesaplar (orn. NIH validation)."""
    all_probs, all_labels = [], []
    loss_meter = AverageMeter()
    criterion = nn.CrossEntropyLoss()
    amp_device_type = "cuda" if device == "cuda" else "cpu"
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast(device_type=amp_device_type, enabled=(use_amp and device == "cuda")):
                logits = forward_fn(images)
                loss = criterion(logits, labels)
            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
            loss_meter.update(loss.item(), images.size(0))
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")
    return {"loss": loss_meter.avg, "auc": auc}


def compute_validation_scores(config: "Config", forward_fn: Callable[[torch.Tensor], torch.Tensor],
                               source_val_loader: DataLoader, target_val_loader: Optional[DataLoader],
                               device: str) -> Dict:
    """
    Her epoch sonunda hem source validation (loss/AUC) hem de -mevcutsa- target
    validation AUC'sini (volume-level, bkz Section 14) hesaplar. Hangisinin model
    selection icin KULLANILACAGI cfg.USE_TARGET_LABELS_FOR_MODEL_SELECTION ile
    belirlenir (bkz Section 12/13).
    """
    source_val_metrics = evaluate_classifier_on_loader(forward_fn, source_val_loader, device, config.USE_AMP)
    target_val_auc = float("nan")
    if target_val_loader is not None:
        target_probs, target_labels = volume_level_predict(forward_fn, target_val_loader, device, config.USE_AMP)
        if len(np.unique(target_labels)) > 1:
            target_val_auc = roc_auc_score(target_labels, target_probs)
    return {
        "source_val_loss": source_val_metrics["loss"],
        "source_val_auc": source_val_metrics["auc"],
        "target_val_auc": target_val_auc,
    }


## 12. Source-only training (Experiment 1)


In [ ]:
def train_source_only(config: "Config", model: nn.Module, train_loader: DataLoader,
                       source_val_loader: DataLoader, target_val_loader: Optional[DataLoader],
                       class_weights: Optional[torch.Tensor] = None) -> Tuple[nn.Module, Dict]:
    """
    Experiment 1 - Source Only / Transfer Learning Baseline.
    Yalnizca NIH CXR train verisiyle egitilir; MosMed goruntuleri bu asamada HIC
    kullanilmaz (ne train ne validation girdisi olarak backbone'a verilmez -
    target_val_loader yalnizca RAPORLAMA/model-selection ICIN CIKTIYA bakmak
    amaciyla kullanilir, egitim gradyanlarina hicbir sekilde girmez).
    """
    device = config.DEVICE
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    amp_device_type = "cuda" if device == "cuda" else "cpu"
    scaler = torch.amp.GradScaler(enabled=(config.USE_AMP and device == "cuda"))

    use_target_for_selection = config.USE_TARGET_LABELS_FOR_MODEL_SELECTION and target_val_loader is not None
    best_score = -float("inf") if use_target_for_selection else float("inf")
    best_state, best_epoch = None, -1

    for epoch in range(config.NUM_EPOCHS_SOURCE_ONLY):
        model.train()
        loss_meter = AverageMeter()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=amp_device_type, enabled=(config.USE_AMP and device == "cuda")):
                logits, _ = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_meter.update(loss.item(), images.size(0))

        model.eval()
        forward_fn = lambda x: model(x)[0]
        scores = compute_validation_scores(config, forward_fn, source_val_loader, target_val_loader, device)
        print(f"[Source-Only][Epoch {epoch + 1}/{config.NUM_EPOCHS_SOURCE_ONLY}] "
              f"train_loss={loss_meter.avg:.4f} | source_val_loss={scores['source_val_loss']:.4f} | "
              f"source_val_auc={scores['source_val_auc']:.4f} | target_val_auc={scores['target_val_auc']:.4f}")

        current_score = scores["target_val_auc"] if use_target_for_selection else scores["source_val_loss"]
        is_best = (current_score > best_score) if use_target_for_selection else (current_score < best_score)
        if is_best and not math.isnan(current_score):
            best_score = current_score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    save_checkpoint(config.SOURCE_ONLY_CKPT_PATH, model, optimizer, best_epoch, config,
                     {"selection_criterion": "target_val_auc" if use_target_for_selection else "source_val_loss",
                      "best_score": best_score})
    return model, {"best_epoch": best_epoch, "best_score": best_score}


## 13. DANN training (Experiment 2)


In [ ]:
def compute_domain_accuracy(domain_logits: torch.Tensor, domain_labels: torch.Tensor) -> float:
    preds = domain_logits.argmax(dim=1)
    return (preds == domain_labels).float().mean().item()


def train_dann(config: "Config", model: nn.Module, source_train_loader: DataLoader,
                target_train_loader: DataLoader, source_val_loader: DataLoader,
                target_val_loader: Optional[DataLoader],
                class_weights: Optional[torch.Tensor] = None) -> Tuple[nn.Module, Dict]:
    """
    Experiment 2 - DANN Domain Adaptation.
    Source orneklerinde classification_loss + domain_loss, target orneklerinde
    YALNIZCA domain_loss hesaplanir; target disease label egitim sirasinda HICBIR
    NOKTADA kullanilmaz (target_train_loader'dan gelen etiket asagida '_' ile
    aciktan atilir). Yalnizca MosMed TRAIN split'i kullanilir; validation/test
    split'leri bu fonksiyona hic verilmez (target_val_loader yalnizca model
    selection/RAPORLAMA icin forward-pass CIKTISINA bakar, egitime katilmaz).

    steps_per_epoch = max(len(source_loader), len(target_loader)) kullanilir ve
    kisa olan loader'in iterator'u tukendiginde YENIDEN BASLATILIR; boylece bir
    epoch, yalnizca source (veya yalnizca target) uzunluguna gore kisitlanmaz ve
    iki domainden daha dengeli ornekleme saglanir.
    """
    device = config.DEVICE
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
    class_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    domain_criterion = nn.CrossEntropyLoss()
    amp_device_type = "cuda" if device == "cuda" else "cpu"
    scaler = torch.amp.GradScaler(enabled=(config.USE_AMP and device == "cuda"))

    steps_per_epoch = max(len(source_train_loader), len(target_train_loader))
    total_steps = steps_per_epoch * config.NUM_EPOCHS_DANN

    use_target_for_selection = config.USE_TARGET_LABELS_FOR_MODEL_SELECTION and target_val_loader is not None
    best_score = -float("inf") if use_target_for_selection else float("inf")
    best_state, best_epoch = None, -1
    global_step = 0

    for epoch in range(config.NUM_EPOCHS_DANN):
        model.train()
        source_iter = iter(source_train_loader)
        target_iter = iter(target_train_loader)

        cls_loss_meter = AverageMeter()
        domain_loss_meter = AverageMeter()
        total_loss_meter = AverageMeter()
        domain_acc_meter = AverageMeter()
        last_lambda = 0.0

        for _ in range(steps_per_epoch):
            try:
                source_images, source_labels = next(source_iter)
            except StopIteration:
                source_iter = iter(source_train_loader)
                source_images, source_labels = next(source_iter)
            try:
                target_images, _ = next(target_iter)  # target disease label KASTEN atiliyor
            except StopIteration:
                target_iter = iter(target_train_loader)
                target_images, _ = next(target_iter)

            source_images = source_images.to(device)
            source_labels = source_labels.to(device)
            target_images = target_images.to(device)

            lambda_p = compute_grl_lambda(global_step, total_steps)
            last_lambda = lambda_p

            bs_source = source_images.size(0)
            bs_target = target_images.size(0)
            domain_labels_source = torch.zeros(bs_source, dtype=torch.long, device=device)
            domain_labels_target = torch.ones(bs_target, dtype=torch.long, device=device)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=amp_device_type, enabled=(config.USE_AMP and device == "cuda")):
                class_logits_source, domain_logits_source, _ = model(source_images, grl_lambda=lambda_p)
                _, domain_logits_target, _ = model(target_images, grl_lambda=lambda_p)

                classification_loss = class_criterion(class_logits_source, source_labels)
                domain_loss = (domain_criterion(domain_logits_source, domain_labels_source)
                               + domain_criterion(domain_logits_target, domain_labels_target))
                total_loss = classification_loss + config.DOMAIN_LOSS_WEIGHT * domain_loss

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            all_domain_logits = torch.cat([domain_logits_source, domain_logits_target], dim=0).detach()
            all_domain_labels = torch.cat([domain_labels_source, domain_labels_target], dim=0)
            domain_acc = compute_domain_accuracy(all_domain_logits, all_domain_labels)

            cls_loss_meter.update(classification_loss.item(), bs_source)
            domain_loss_meter.update(domain_loss.item(), bs_source + bs_target)
            total_loss_meter.update(total_loss.item(), bs_source)
            domain_acc_meter.update(domain_acc, bs_source + bs_target)
            global_step += 1

        model.eval()
        forward_fn = lambda x: model(x, grl_lambda=0.0)[0]
        scores = compute_validation_scores(config, forward_fn, source_val_loader, target_val_loader, device)

        print(f"[DANN][Epoch {epoch + 1}/{config.NUM_EPOCHS_DANN}] "
              f"cls_loss={cls_loss_meter.avg:.4f} | domain_loss={domain_loss_meter.avg:.4f} | "
              f"total_loss={total_loss_meter.avg:.4f} | domain_acc={domain_acc_meter.avg:.4f} (ideal ~0.50) | "
              f"grl_lambda={last_lambda:.4f} | source_val_loss={scores['source_val_loss']:.4f} | "
              f"target_val_auc={scores['target_val_auc']:.4f}")
        # NOT: domain_acc'nin ~0.50'ye yaklasmasi TEK BASINA basarili domain alignment
        # anlamina gelmez -- backbone, class-discriminative bilgiyi feda ederek de
        # domain-invariant (ayirt edici olmayan) feature'lar ogrenebilir (negative
        # transfer). Bu yuzden target_val_auc/target_val performansi ile BIRLIKTE
        # yorumlanmalidir (bkz. Section 17/19).

        current_score = scores["target_val_auc"] if use_target_for_selection else scores["source_val_loss"]
        is_best = (current_score > best_score) if use_target_for_selection else (current_score < best_score)
        if is_best and not math.isnan(current_score):
            best_score = current_score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    save_checkpoint(config.DANN_CKPT_PATH, model, optimizer, best_epoch, config,
                     {"selection_criterion": "target_val_auc" if use_target_for_selection else "source_val_loss",
                      "best_score": best_score})
    return model, {"best_epoch": best_epoch, "best_score": best_score}


## 14. Volume-level CT inference


In [ ]:
def volume_level_predict(forward_fn: Callable[[torch.Tensor], torch.Tensor], loader: DataLoader,
                          device: str, use_amp: bool) -> Tuple[np.ndarray, np.ndarray]:
    """
    MosMedCTEvalDataset tabanli bir loader uzerinde slice-level tahmin uretir ve
    volume-level probability = mean(slice_probabilities) olarak agregasyon yapar.
    Dondurulen (volume_probs, volume_labels), yalnizca en az bir slice'i skorlanmis
    volume'lari icerir.
    """
    dataset = loader.dataset
    num_volumes = dataset.num_volumes()
    prob_sums = np.zeros(num_volumes, dtype=np.float64)
    slice_counts = np.zeros(num_volumes, dtype=np.int64)
    volume_labels = np.full(num_volumes, -1, dtype=np.int64)
    amp_device_type = "cuda" if device == "cuda" else "cpu"

    with torch.no_grad():
        for images, labels, v_indices in loader:
            images = images.to(device)
            with torch.amp.autocast(device_type=amp_device_type, enabled=(use_amp and device == "cuda")):
                logits = forward_fn(images)
            probs = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            labels_np = labels.numpy()
            v_indices_np = v_indices.numpy()
            for p, l, v in zip(probs, labels_np, v_indices_np):
                prob_sums[v] += p
                slice_counts[v] += 1
                volume_labels[v] = l

    valid_mask = slice_counts > 0
    volume_probs = np.zeros(num_volumes, dtype=np.float64)
    volume_probs[valid_mask] = prob_sums[valid_mask] / slice_counts[valid_mask]
    return volume_probs[valid_mask], volume_labels[valid_mask]


## 15. Evaluation metrics


In [ ]:
def find_optimal_threshold_youden(labels: np.ndarray, probs: np.ndarray) -> float:
    """Youden's J istatistigi (TPR - FPR) maksimize eden threshold'u dondurur."""
    fpr, tpr, thresholds = roc_curve(labels, probs)
    youden_j = tpr - fpr
    best_idx = int(np.argmax(youden_j))
    return float(thresholds[best_idx])


def compute_binary_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float) -> Dict:
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return {
        "threshold": threshold,
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),
        "pr_auc": average_precision_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall_sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(labels, preds, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "confusion_matrix": cm,
    }


def _print_metrics(metrics: Dict):
    for key, value in metrics.items():
        if key == "confusion_matrix":
            print(f"  confusion_matrix:\n{value}")
        elif isinstance(value, (int, float, np.floating)):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")


def evaluate_full_report(val_labels: np.ndarray, val_probs: np.ndarray,
                          test_labels: np.ndarray, test_probs: np.ndarray, model_name: str) -> Dict:
    """
    Threshold, YALNIZCA validation set kullanilarak (Youden's J) optimize edilir;
    test seti threshold optimizasyonu icin ASLA kullanilmaz. Iki sonuc raporlanir:
    threshold=0.5 ve validation-optimal threshold.
    """
    metrics_default = compute_binary_metrics(test_labels, test_probs, threshold=0.5)
    optimal_threshold = find_optimal_threshold_youden(val_labels, val_probs)
    metrics_optimal = compute_binary_metrics(test_labels, test_probs, threshold=optimal_threshold)

    print(f"\n=== {model_name} | MosMed TEST (threshold=0.5) ===")
    _print_metrics(metrics_default)
    print(f"\n=== {model_name} | MosMed TEST (validation-optimal threshold={optimal_threshold:.4f}) ===")
    _print_metrics(metrics_optimal)

    return {"threshold_0.5": metrics_default, "threshold_optimal": metrics_optimal}


## 16. Feature extraction


In [ ]:
def extract_features(feature_fn: Callable[[torch.Tensor], torch.Tensor], loader: DataLoader,
                      device: str, max_samples: int, use_amp: bool) -> np.ndarray:
    """
    Domain discrepancy analizi (MMD/CORAL) icin backbone feature'larini cikarir.
    Karsilastirilabilirlik icin her domainden max_samples kadar ornek kullanilir;
    bu ornekler NIH source validation/test ve MosMed target test split'lerinden,
    egitimde dogrudan kullanilan batch'lerden BAGIMSIZ ayri bir forward-pass ile
    elde edilir.
    """
    features_list = []
    collected = 0
    amp_device_type = "cuda" if device == "cuda" else "cpu"
    with torch.no_grad():
        for batch in loader:
            images = batch[0].to(device)
            with torch.amp.autocast(device_type=amp_device_type, enabled=(use_amp and device == "cuda")):
                feats = feature_fn(images)
            features_list.append(feats.detach().cpu().numpy())
            collected += images.size(0)
            if collected >= max_samples:
                break
    features = np.concatenate(features_list, axis=0)
    return features[:max_samples]


## 17. MMD / CORAL


In [ ]:
def compute_mmd(x: np.ndarray, y: np.ndarray, gamma: Optional[float] = None) -> float:
    """Gaussian-kernel Maximum Mean Discrepancy (biased estimator). gamma=None -> median heuristic."""
    x_t = torch.from_numpy(x).float()
    y_t = torch.from_numpy(y).float()

    if gamma is None:
        combined = torch.cat([x_t, y_t], dim=0)
        dists = torch.cdist(combined, combined, p=2)
        nonzero = dists[dists > 0]
        median_dist = torch.median(nonzero) if nonzero.numel() > 0 else torch.tensor(1.0)
        gamma = 1.0 / (2 * (median_dist ** 2) + 1e-8)

    def gaussian_kernel(a, b, g):
        dist_sq = torch.cdist(a, b, p=2) ** 2
        return torch.exp(-g * dist_sq)

    k_xx = gaussian_kernel(x_t, x_t, gamma).mean()
    k_yy = gaussian_kernel(y_t, y_t, gamma).mean()
    k_xy = gaussian_kernel(x_t, y_t, gamma).mean()
    mmd = k_xx + k_yy - 2 * k_xy
    return float(mmd.item())


def compute_coral(x: np.ndarray, y: np.ndarray) -> float:
    """CORAL distance: kaynak/hedef feature kovaryans matrisleri arasi normalize Frobenius-norm farki."""
    d = x.shape[1]
    x_centered = x - x.mean(axis=0, keepdims=True)
    y_centered = y - y.mean(axis=0, keepdims=True)
    cov_x = (x_centered.T @ x_centered) / (x.shape[0] - 1 + 1e-8)
    cov_y = (y_centered.T @ y_centered) / (y.shape[0] - 1 + 1e-8)
    coral = np.sum((cov_x - cov_y) ** 2) / (4.0 * d * d)
    return float(coral)


## 18. Visualization


In [ ]:
def plot_roc_curves(results: Dict[str, Tuple[np.ndarray, np.ndarray]], save_path: str):
    plt.figure(figsize=(6, 6))
    for model_name, (labels, probs) in results.items():
        if len(np.unique(labels)) < 2:
            print(f"[UYARI] {model_name} icin tek sinif mevcut, ROC atlaniyor.")
            continue
        fpr, tpr, _ = roc_curve(labels, probs)
        auc_score = roc_auc_score(labels, probs)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_score:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("MosMed TEST ROC Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


def plot_feature_space(source_features: np.ndarray, target_features: np.ndarray, title: str,
                        save_path: str, seed: int, method: str = "tsne"):
    combined = np.concatenate([source_features, target_features], axis=0)
    domain_labels = np.array([0] * len(source_features) + [1] * len(target_features))

    if method == "tsne":
        from sklearn.manifold import TSNE
        perplexity = min(30, max(5, len(combined) // 10))
        reducer = TSNE(n_components=2, random_state=seed, init="pca", perplexity=perplexity)
        embedding = reducer.fit_transform(combined)
    elif method == "umap":
        try:
            import umap
        except ImportError:
            print("[UYARI] umap-learn kurulu degil, t-SNE'ye geri donuluyor.")
            return plot_feature_space(source_features, target_features, title, save_path, seed, method="tsne")
        reducer = umap.UMAP(n_components=2, random_state=seed)
        embedding = reducer.fit_transform(combined)
    else:
        raise ValueError(f"Desteklenmeyen boyut indirgeme yontemi: {method}")

    plt.figure(figsize=(6, 6))
    plt.scatter(embedding[domain_labels == 0, 0], embedding[domain_labels == 0, 1],
                label="Source (NIH)", alpha=0.6, s=15)
    plt.scatter(embedding[domain_labels == 1, 0], embedding[domain_labels == 1, 1],
                label="Target (MosMed)", alpha=0.6, s=15)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


## 19. Experiment runner


In [ ]:
def run_experiment_pipeline(config: "Config") -> Dict:
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    device = config.DEVICE

    print("=" * 80); print("STEP 1: Dataset discovery"); print("=" * 80)
    nih_image_index = discover_nih_images(config.NIH_ROOT)
    nih_df = build_nih_labels(config.NIH_CSV, nih_image_index, config.INCLUDE_INFILTRATION)
    mosmed_df = discover_mosmed_volumes(config.MOSMED_ROOT)

    print("\n" + "=" * 80); print("STEP 2: Dataset split"); print("=" * 80)
    nih_train_df, nih_val_df = split_nih(nih_df, config.NIH_VAL_RATIO, config.SEED)
    mosmed_train_df, mosmed_val_df, mosmed_test_df = split_mosmed(
        mosmed_df, config.TARGET_TRAIN_RATIO, config.TARGET_VAL_RATIO, config.TARGET_TEST_RATIO, config.SEED
    )

    print("\n" + "=" * 80); print("STEP 3: DataLoaders"); print("=" * 80)
    nih_train_loader, nih_val_loader = build_nih_dataloaders(nih_train_df, nih_val_df, config)
    mosmed_train_loader, mosmed_val_loader, mosmed_test_loader = build_mosmed_dataloaders(
        mosmed_train_df, mosmed_val_df, mosmed_test_df, config
    )

    class_weights = None
    if not config.USE_WEIGHTED_SAMPLER:
        class_weights = compute_class_weights_tensor(nih_train_df["label"].tolist())
        print(f"[NIH] USE_WEIGHTED_SAMPLER=False -> weighted CrossEntropyLoss, class_weights={class_weights.tolist()}")

    target_val_loader_for_selection = mosmed_val_loader if config.USE_TARGET_LABELS_FOR_MODEL_SELECTION else None
    if config.USE_TARGET_LABELS_FOR_MODEL_SELECTION:
        print("\n[METODOLOJI NOTU] Model selection MosMed VALIDATION AUC uzerinden yapiliyor. Target "
              "etiketleri YALNIZCA checkpoint secimi icin kullaniliyor; egitim (gradyan guncellemesi) "
              "target-label-free kalmaya devam ediyor. Tamamen unsupervised protokol icin "
              "USE_TARGET_LABELS_FOR_MODEL_SELECTION=False kullanin.")
    else:
        print("\n[METODOLOJI NOTU] Model selection NIH source validation loss uzerinden yapiliyor "
              "(tamamen unsupervised protokol; target hicbir asamada kullanilmiyor).")

    print("\n" + "=" * 80); print("STEP 4: Experiment 1 - Source Only Baseline"); print("=" * 80)
    source_only_model = SourceOnlyModel(config)
    source_only_model, source_only_info = train_source_only(
        config, source_only_model, nih_train_loader, nih_val_loader,
        target_val_loader_for_selection, class_weights,
    )

    print("\n" + "=" * 80); print("STEP 5: Experiment 2 - DANN Domain Adaptation"); print("=" * 80)
    dann_model = DANNModel(config)
    dann_model, dann_info = train_dann(
        config, dann_model, nih_train_loader, mosmed_train_loader,
        nih_val_loader, target_val_loader_for_selection, class_weights,
    )

    print("\n" + "=" * 80); print("STEP 6: MosMed TEST uzerinde volume-level degerlendirme"); print("=" * 80)
    source_only_model.eval()
    dann_model.eval()
    source_only_forward = lambda x: source_only_model(x)[0]
    dann_forward = lambda x: dann_model(x, grl_lambda=0.0)[0]

    val_probs_so, val_labels_so = volume_level_predict(source_only_forward, mosmed_val_loader, device, config.USE_AMP)
    test_probs_so, test_labels_so = volume_level_predict(source_only_forward, mosmed_test_loader, device, config.USE_AMP)
    val_probs_dann, val_labels_dann = volume_level_predict(dann_forward, mosmed_val_loader, device, config.USE_AMP)
    test_probs_dann, test_labels_dann = volume_level_predict(dann_forward, mosmed_test_loader, device, config.USE_AMP)

    report_source_only = evaluate_full_report(val_labels_so, val_probs_so, test_labels_so, test_probs_so, "Source-Only")
    report_dann = evaluate_full_report(val_labels_dann, val_probs_dann, test_labels_dann, test_probs_dann, "DANN")

    plot_roc_curves(
        {"Source-Only": (test_labels_so, test_probs_so), "DANN": (test_labels_dann, test_probs_dann)},
        os.path.join(config.OUTPUT_DIR, "roc_curves.png"),
    )

    print("\n" + "=" * 80); print("STEP 7: Domain alignment analizi (MMD / CORAL)"); print("=" * 80)
    feature_eval_transform = build_eval_transform(config)
    nih_feat_loader = DataLoader(NIHChestXrayDataset(nih_val_df, feature_eval_transform),
                                  batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
    mosmed_feat_loader = DataLoader(MosMedCTEvalDataset(mosmed_test_df, feature_eval_transform, config),
                                     batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

    so_feat_fn = lambda x: source_only_model.backbone(x)
    dann_feat_fn = lambda x: dann_model.backbone(x)

    source_feats_so = extract_features(so_feat_fn, nih_feat_loader, device, config.FEATURE_EXTRACTION_N_SAMPLES, config.USE_AMP)
    target_feats_so = extract_features(so_feat_fn, mosmed_feat_loader, device, config.FEATURE_EXTRACTION_N_SAMPLES, config.USE_AMP)
    source_feats_dann = extract_features(dann_feat_fn, nih_feat_loader, device, config.FEATURE_EXTRACTION_N_SAMPLES, config.USE_AMP)
    target_feats_dann = extract_features(dann_feat_fn, mosmed_feat_loader, device, config.FEATURE_EXTRACTION_N_SAMPLES, config.USE_AMP)

    mmd_so, coral_so = compute_mmd(source_feats_so, target_feats_so), compute_coral(source_feats_so, target_feats_so)
    mmd_dann, coral_dann = compute_mmd(source_feats_dann, target_feats_dann), compute_coral(source_feats_dann, target_feats_dann)

    print(f"[Source-Only] MMD={mmd_so:.6f} | CORAL={coral_so:.6f}")
    print(f"[DANN]        MMD={mmd_dann:.6f} | CORAL={coral_dann:.6f}")
    print("\n[METODOLOJI NOTU] MMD/CORAL'in dusmesi TEK BASINA daha iyi adaptasyon anlamina gelmez. "
          "Ornegin MMD dusuyorken target AUC de dusuyorsa, domain alignment gerceklesmis ancak "
          "class-discriminative bilgi kaybolmus (negative transfer) olabilir. Bu yuzden MMD/CORAL "
          "her zaman target AUC/performans ile BIRLIKTE yorumlanmalidir.")

    try:
        plot_feature_space(source_feats_so, target_feats_so, "Source-Only Feature Space (t-SNE)",
                            os.path.join(config.OUTPUT_DIR, "tsne_source_only.png"), config.SEED, method="tsne")
        plot_feature_space(source_feats_dann, target_feats_dann, "DANN Feature Space (t-SNE)",
                            os.path.join(config.OUTPUT_DIR, "tsne_dann.png"), config.SEED, method="tsne")
    except Exception as e:
        print(f"[UYARI] t-SNE gorsellestirmesi basarisiz oldu: {e}")

    return {
        "source_only_model": source_only_model, "dann_model": dann_model,
        "report_source_only": report_source_only, "report_dann": report_dann,
        "mmd_so": mmd_so, "coral_so": coral_so, "mmd_dann": mmd_dann, "coral_dann": coral_dann,
    }


## 20. Final comparison


In [ ]:
def build_results_table(pipeline_results: Dict, config: "Config") -> pd.DataFrame:
    rows = []
    for model_name, report_key, mmd_key, coral_key in [
        ("Source Only", "report_source_only", "mmd_so", "coral_so"),
        ("DANN", "report_dann", "mmd_dann", "coral_dann"),
    ]:
        m_opt = pipeline_results[report_key]["threshold_optimal"]
        m_def = pipeline_results[report_key]["threshold_0.5"]
        rows.append({
            "Model": model_name,
            "AUC": m_opt["roc_auc"],
            "PR-AUC": m_opt["pr_auc"],
            "ACC (thr=0.5)": m_def["accuracy"],
            "ACC (thr=opt)": m_opt["accuracy"],
            "F1 (thr=0.5)": m_def["f1"],
            "F1 (thr=opt)": m_opt["f1"],
            "Sensitivity (thr=opt)": m_opt["recall_sensitivity"],
            "Specificity (thr=opt)": m_opt["specificity"],
            "Balanced-ACC (thr=opt)": m_opt["balanced_accuracy"],
            "Optimal-Threshold": m_opt["threshold"],
            "MMD": pipeline_results[mmd_key],
            "CORAL": pipeline_results[coral_key],
        })
    results_df = pd.DataFrame(rows)

    print("\n" + "=" * 80); print("FINAL EXPERIMENT COMPARISON"); print("=" * 80)
    print(results_df.to_string(index=False))

    results_df.to_csv(config.RESULTS_CSV_PATH, index=False)
    print(f"\n[Sonuclar kaydedildi] {config.RESULTS_CSV_PATH}")
    return results_df


### Pipeline'i calistir

Asagidaki hucre tum deneyi (veri kesfi -> split -> Source-Only egitimi -> DANN
egitimi -> MosMed TEST degerlendirmesi -> MMD/CORAL analizi -> sonuc tablosu)
uctan uca calistirir. `Config` icindeki `NIH_ROOT`, `NIH_CSV`, `MOSMED_ROOT`
yollarini kendi Kaggle input path'lerinize gore guncelleyin.


In [ ]:
pipeline_results = run_experiment_pipeline(cfg)
final_results_df = build_results_table(pipeline_results, cfg)
final_results_df
